# Hypnos reference notebook: model-divergence & verification

**NOT FOR CLINICAL USE.** Research / education / simulation only.

This notebook is executed in CI (via `nbmake`) so it cannot rot. It reproduces the headline
model-divergence comparison and shows the verification workflow. It depends only on the
`hypnos` package and NumPy; plotting is optional (skipped if matplotlib is absent).

In [ ]:
import numpy as np
import hypnos

ds = hypnos.load()
summary = hypnos.summary(ds)
print('models', summary['n_models'], '| drugs', summary['n_drugs'],
      '| kernels', summary['kernels_implemented'])
print('by subsystem:', summary['by_subsystem'])
assert hypnos.validate_dataset(ds) == [], 'dataset must validate'

## Model divergence: same patient, same dose, different models

For an elderly patient, the eligible propofol models stay in-envelope but disagree substantially
on the predicted effect-site concentration. That disagreement is the model-selection risk Hypnos
exists to make visible.

In [ ]:
patient = dict(age=72, weight=60, height=162, sex='F')
schedule = [('bolus', 0.0, '2 mg/kg'), ('infusion', 0.0, '6 mg/kg/h')]
t = np.linspace(0, 60, 361)

cmp = hypnos.compare(ds, drug='propofol', patient=patient, schedule=schedule, t=t)
for r in cmp.included:
    print(f"{r.model_id:42s} tier {r.tier}  Ce peak {r.ce_peak:.2f} ug/mL")
for u in cmp.unavailable:
    print(f"{u['model_id']:42s} (kernel pending)")
d = cmp.divergence['ce']
print(f"\npeak effect-site divergence: {d['max_abs']:.2f} ug/mL ({100*d['max_rel']:.0f}%)")
assert d['max_abs'] > 0.5  # the models genuinely disagree

In [ ]:
# Optional plot (skipped if matplotlib is not installed; the curated figure lives in docs/images/).
try:
    import matplotlib.pyplot as plt
    for r in cmp.included:
        plt.plot(t, r.ce, label=f"{r.model_id.split('.')[-1]} (tier {r.tier})")
    plt.xlabel('time (min)'); plt.ylabel('effect-site (ug/mL)')
    plt.title('Propofol effect-site divergence  -  NOT FOR CLINICAL USE')
    plt.legend(); plt.show()
except ImportError:
    print('matplotlib not installed; see docs/images/divergence.png')

## Envelope enforcement & explicit extrapolation labeling

Out-of-envelope requests are tiered down to D and named. For a child, only the pediatric
Paedfusor model stays in-envelope; the adult models are greyed out as *pediatric extrapolations*.

In [ ]:
child = dict(age=6, weight=20, height=115, sex='M')
ped = hypnos.compare(ds, drug='propofol', patient=child,
                     schedule=[('bolus', 0.0, '2 mg/kg')], t=t)
print('in-envelope:', [r.model_id.split('.')[-1] for r in ped.included])
for e in ped.excluded:
    reason = next((w for w in e['reasons'] if 'EXTRAPOLATION' in w), e['reasons'][0])
    print(f"greyed: {e['model_id'].split('.')[-1]:16s} tier {e['tier']}  ({reason[:70]}...)")
assert {r.model_id for r in ped.included} == {'hypnotics_iv.propofol.paedfusor_2005'}

## Verification status: the single highest-leverage contribution

Every model is `unverified` until a human confirms it against the source PDF. `hypnos status`
reports coverage and what to verify next; `hypnos verify <id>` prints the field-by-field checklist.

In [ ]:
vs = hypnos.verification_summary(ds)
print('verified fraction:', round(vs['verified_fraction'], 2))
print('start here:')
for item in vs['next_to_verify']:
    print(f"  {item['model_id']:46s} tier {item['tier']}  cite {item['citation']}")

mv = hypnos.model_verification(ds, 'hypnotics_iv.propofol.schnider_1998')
print(f"\n{mv.model_id}: {mv.n_items} items to confirm against doi:{mv.doi}")
cov = [it for it in mv.checklist if it.group == 'covariate']
print('covariate equations to double-check (where transcription errors hide):')
for it in cov[:3]:
    print(f"  - {it.label}: {it.value}")